# 0 - From Text to Numbers (and Back)

A model can't read letters, so we need to convert letters to numbers:
* Encode: text → numbers
* Decode: numbers → text

Vocabulary: a list of characters where each character gets a unique integer ID
* Encoding looks up IDs
* Decoding looks them back up. 

(`text → numbers → text`)

A teeny-tiny toy vocab before the real Shakespeare vocab in Section 2

In [1]:
# Toy vocabulary: characters in one sentence
vocab = sorted(set('hello world'))
print(vocab)
print(len(vocab), 'tokens')                      # "vocab_size"

# Two lookup tables
stoi = {ch: i for i, ch in enumerate(vocab)}     # string to int
itos = {i: ch for i, ch in enumerate(vocab)}     # int to string
print(stoi)
print(itos)

encode = lambda s: [stoi[ch] for ch in s]        # text -> list of ints
decode = lambda l: ''.join(itos[i] for i in l)   # list of ints -> text

ids = encode('hello')
print('text   ->', ids)                          # text -> numbers
print('numbers->', decode(ids))                  # numbers -> text

[' ', 'd', 'e', 'h', 'l', 'o', 'r', 'w']
8 tokens
{' ': 0, 'd': 1, 'e': 2, 'h': 3, 'l': 4, 'o': 5, 'r': 6, 'w': 7}
{0: ' ', 1: 'd', 2: 'e', 3: 'h', 4: 'l', 5: 'o', 6: 'r', 7: 'w'}
text   -> [3, 2, 4, 4, 5]
numbers-> hello


# 1 - Bigram Language Model

The simplest possible language model
* Predicts the next token based on *only the current token*
* Doesn't look at history (deliberately)
    * GPT would consider all previous tokens
* "*Bi*gram": considers sequences of two tokens (bi): the current one and the predicted next one.
* Learns a probability table: given token X, what is the probability distribution over the next token?
    * $P(\text{next} | \text{current})$ - one row per token in the vocab

---
Logits: probability scores
* Not probabilites yet cause haven't sum to `1`
* Convert to probabilites with `softmax` later
* One score per possible output token `vocab_size`

Embeddings: token representation
* Vector with embedding dimension `n_embd`
* Special case here: use embedding table to store logits for simplification
    * `n_embd = vocab_size`
---
Shapes:
* `B` for `batch_size`
* `T` for time dimension `block_size`
* `C` for channels `n_embd`

`idx` = a tensor of token IDs

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# Define a simple Bigram Model
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):                                # (B, T)
        logits = self.token_embedding_table(idx)                         # (B, T, C)

        if targets is None:
            loss = None
        else: 
            B, T, C = logits.shape
            logits = logits.view(B*T, C)                                 # (B*T, C)
            targets = targets.view(B*T)                                  # (B*T, )
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):                             # (B, T)

        for _ in range(max_new_tokens):
            logits, _ = self(idx)                                        # (B, T, C)
            logits = logits[:, -1, :]                                    # (B, C)
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)           # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)                      # (B, T+1)
        return idx


> A Lookup Table
* Creates a learnable 2D weight matrix -> Bigram probability table
    * `nn.Embedding(vocab_size, vocab_size)`
* A lookup table. Looks up the current token's row 
    * `logits = self.token_embedding_table(idx)`: (B, T) -> (B, T, C)

> Find the next token with the highest probability
1. Take only the last time step (*bi*gram only cares about the last token)
    * `logits[:, -1, :]` 
2. Normalize with `F.softmax`
    * Probability score -> probability
3. Sample from the distribution with `torch.multinomial`
    * Varied outputs, probabilistic

> Loss calculation with `F.cross_entropy`
* `(N, C, d1, ...dk)` or `(B, C, T)`: standard for sequential data
    * `(batch, classes, height, width)`
* `(N, C)` or `(B*T, C)` for NLP looks more natural

In [3]:
if torch.backends.mps.is_available():
    device = 'mps'
elif torch.cuda.is_available():
    device = 'cuda'
else: 
    device = 'cpu'

print(device)

# Test the Bigram model
test_vocab_size = 60              # Randomly set
test_model = BigramLanguageModel(test_vocab_size)
test_model = test_model.to(device)
print(type(test_model), test_model.token_embedding_table.weight.device)
print(test_model)

test_idx = torch.zeros((1, 1), dtype=torch.long).to(device)
print(test_idx, test_idx.dtype, test_idx.device)

max_new_tokens = 100

new_token_idx = test_model.generate(test_idx, max_new_tokens)
print(new_token_idx, new_token_idx.device)

mps
<class '__main__.BigramLanguageModel'> mps:0
BigramLanguageModel(
  (token_embedding_table): Embedding(60, 60)
)
tensor([[0]], device='mps:0') torch.int64 mps:0
tensor([[ 0, 51, 50, 58, 17, 26, 33, 22, 36, 21, 34, 41, 29, 10, 39,  5, 49, 28,
         43, 20, 22, 36, 35,  4, 38, 42,  4, 25,  8, 17, 19, 54, 42, 41, 51, 10,
         22, 18, 38, 27, 30, 21, 33, 56, 26, 32, 45, 48, 23, 25, 12, 20, 51, 22,
         52, 43, 50,  4, 30, 50, 18, 50, 18, 46, 21, 35, 19, 54,  0, 21, 34, 15,
         19, 44, 25,  6, 25, 48, 13, 50, 57, 32, 39, 33, 18, 12, 36,  3, 39, 27,
         50, 32, 25, 27, 39, 27, 14, 58, 25, 48,  2]], device='mps:0') mps:0


# 2 - Tokenization

Tokenization: the process of turning string to integers

1. Split text into 'tokens' and get a vocabulary, a list of known tokens
2. Map each token to an integer, a unique integer ID

Character-level: split texts into single characters  -> small vocabulary, long sequence


In [4]:
# Download data to the data folder
# !pwd
# !wget -O ../data/tinyshakespear.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


In [5]:
# Prepare data
with open('../data/tinyshakespear.txt', 'r', encoding='UTF-8') as f:
    text = f.read()
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [6]:
# Vocab - every character in tiny Shakespeare:
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(type(chars))
print(chars)
print(vocab_size)

<class 'list'>
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


In [7]:
# char <-> int mapping
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}

print(type(stoi))
print(stoi)
print(type(itos))
print(itos)

<class 'dict'>
{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
<class 'dict'>
{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: '

In [8]:
# str <-> list of int
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("Hello"))
print(decode(encode("Hello")))

[20, 43, 50, 50, 53]
Hello


In [9]:
# Now we can translate the output tokens with `decode`
# and talk gibberish with the Bigram model

# -----------------------------
# For reference
# test_vocab_size = 60      # Randomly set
# test_model = BigramLanguageModel(test_vocab_size)
# test_model = test_model.to(device)
# print(type(test_model), test_model.token_embedding_table.weight.device)
# print(test_model)

# test_idx = torch.zeros((1, 1), dtype=torch.long).to(device)
# print(test_idx, test_idx.dtype, test_idx.device)

# max_new_tokens = 100

# new_token_idx = test_model.generate(test_idx, max_new_tokens)
# print(new_token_idx, new_token_idx.device)
# -----------------------------

new_tokens = decode(new_token_idx[0].tolist())
print(new_tokens)


mltENUJXIVcQ:a'kPeHJXW&Zd&M.EGpdcm:JFZORIUrNTgjKM?HmJnel&RlFlFhIWGp
IVCGfM,MjAlsTaUF?X$aOlTMOaOBtMj!


# 3 - Processing Data
1. Training, validation split
    * Need to evaluate loss during training
2. Feed the model with **batches** of data

In [10]:
# Dataset and train-val Split
# data = torch.tensor(encode(text), dtype=torch.long)
data = torch.tensor(encode(text))

print(type(data), data.dtype)
print(data[:10])
print(type(text))
print(text[:10])

n_train = int(0.9*len(data))
train_data = data[:n_train]
val_data = data[n_train:]

print(n_train)
print(len(train_data), type(train_data))
print(train_data[:10])
print(len(val_data), type(val_data))
print(val_data[:10])

<class 'torch.Tensor'> torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])
<class 'str'>
First Citi
1003854
1003854 <class 'torch.Tensor'>
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])
111540 <class 'torch.Tensor'>
tensor([12,  0,  0, 19, 30, 17, 25, 21, 27, 10])


## Batching

DON'T load the ENTIRE text into a model at once
* Computational expensive, prohibitive
* Only works with chunks of data (batches of data)

DON'T move the ENTIRE text to GPU
* Out-Of-Memory Warning

Use batches of data

In [11]:
batch_size = 4     # batch dimension: process 4 independent sequences in parallel
block_size = 8     # max content length; each sequence is 8 tokens long

# Sliding window of context
# Target shifted by 1
x = train_data[: block_size]        # input:  token 0-7    
y = train_data[1: block_size+1]     # target: token 1-8


# Time dimension: from one character to the next
for t in range(block_size):
    context = x[: t+1]
    target = y[t]
    print(f"When input is {context} the target: {target}")

When input is tensor([18]) the target: 47
When input is tensor([18, 47]) the target: 56
When input is tensor([18, 47, 56]) the target: 57
When input is tensor([18, 47, 56, 57]) the target: 58
When input is tensor([18, 47, 56, 57, 58]) the target: 1
When input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
When input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
When input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [12]:
# Get random small batch of data of input x and targets y
def get_batch(split):
    data = train_data if split == 'train' else val_data

    # Get `batch_size` random indices from `0` to `len(data)-block_size`
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    xb = torch.stack([data[i : i+block_size] for i in ix])
    yb = torch.stack([data[i+1 : i+block_size+1] for i in ix])

    return xb, yb

test_xb, test_yb = get_batch('train')

# Not moved to device yet
print(type(test_xb), test_xb.shape, test_xb.device)
print(type(test_yb), test_yb.shape, test_yb.device)


<class 'torch.Tensor'> torch.Size([4, 8]) cpu
<class 'torch.Tensor'> torch.Size([4, 8]) cpu


# 4 - Training

In [13]:
# Initialise the model
model = BigramLanguageModel(vocab_size)
model = model.to(device)

print(vocab_size)
print(model)

# Example forward pass 
test_xb, test_yb = test_xb.to(device), test_yb.to(device)
logits, loss = model(test_xb, test_yb)
print(logits.shape)                                              # (B*T, C) = (4*8, 65)
print(loss)

# Example sampling
out_idx = model.generate(
    idx = torch.zeros((1, 1), dtype=torch.long, device=device),  # start (B, T) = (1, 1)
    max_new_tokens=100
    )
print(out_idx.shape)                                             # out (B, T+1) = (1, 1)

print(decode(out_idx[0].tolist()))


65
BigramLanguageModel(
  (token_embedding_table): Embedding(65, 65)
)
torch.Size([32, 65])
tensor(4.9896, device='mps:0', grad_fn=<NllLossBackward0>)
torch.Size([1, 101])

:ign;jaKEtB.SK'.XsAe niev-avwUAvci. ciOqb. HfYRvGwooub'.ptNC,rnnZAl&sBrTPo?rp$Bwoi.xSEBdmGM:jGAlnkMz


In [14]:
# Helper function - visualize loss
eval_iters = 200               # calculate the average loss of 200 iterations

@torch.no_grad()
def estimate_loss(eval_iters):
    out = {}
    model.eval()

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x_eval, y_eval = get_batch(split)
            x_eval, y_eval = x_eval.to(device), y_eval.to(device)
            _, loss = model(x_eval, y_eval)
            losses[k] = loss.item()
        out[split] = losses.mean()
    
    model.train()
    return out

# Test `estimate_loss`
# No training happening here, just checking the function
for i in range(1000):
    if i % 200 == 0:
        out = estimate_loss(eval_iters)
        print(f"Training loss: {out['train']:.4f} | Evaluation loss: {out['val']:.4f}")

Training loss: 4.6838 | Evaluation loss: 4.6958
Training loss: 4.6851 | Evaluation loss: 4.7073
Training loss: 4.6561 | Evaluation loss: 4.6871
Training loss: 4.6701 | Evaluation loss: 4.6908
Training loss: 4.6598 | Evaluation loss: 4.7018


In [15]:
# Training loop

# Set optimizer
learning_rate = 1e-3
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

max_iters = 5000
eval_interval = 200        # Check train and val loss every 200 intervals

for iter in range(max_iters):

    if iter % eval_interval == 0:
        out = estimate_loss(eval_iters)
        print(f"Training loss: {out['train']:.4f} | Eval loss: {out['val']:.4f}")

    # Get batch
    xb, yb = get_batch('train')
    xb, yb = xb.to(device), yb.to(device)

    # Train
    logits, loss = model(xb, yb)      # forward pass: calculate prediction and loss
    optimizer.zero_grad()             # start fresh from the previous step
    loss.backward()                   # compute gradients
    optimizer.step()                  # update weights


Training loss: 4.6640 | Eval loss: 4.6836
Training loss: 4.5327 | Eval loss: 4.5442
Training loss: 4.4112 | Eval loss: 4.4118
Training loss: 4.2475 | Eval loss: 4.2735
Training loss: 4.1306 | Eval loss: 4.1689
Training loss: 3.9990 | Eval loss: 4.0405
Training loss: 3.8933 | Eval loss: 3.9416
Training loss: 3.7807 | Eval loss: 3.8347
Training loss: 3.7083 | Eval loss: 3.7417
Training loss: 3.6162 | Eval loss: 3.6508
Training loss: 3.5367 | Eval loss: 3.5808
Training loss: 3.4486 | Eval loss: 3.4887
Training loss: 3.3665 | Eval loss: 3.4200
Training loss: 3.3360 | Eval loss: 3.3313
Training loss: 3.2738 | Eval loss: 3.2927
Training loss: 3.2201 | Eval loss: 3.2356
Training loss: 3.1356 | Eval loss: 3.1942
Training loss: 3.1025 | Eval loss: 3.1329
Training loss: 3.0724 | Eval loss: 3.0765
Training loss: 3.0190 | Eval loss: 3.0664
Training loss: 2.9758 | Eval loss: 3.0278
Training loss: 2.9324 | Eval loss: 2.9972
Training loss: 2.9068 | Eval loss: 2.9325
Training loss: 2.8934 | Eval loss:

In [16]:
# Generate from the trained model 
# arguably better gibberish 
print(decode(model.generate(torch.zeros((1, 1), dtype=torch.long).to(device), max_new_tokens)[0].tolist()))


UVCKais vFkl, atev
TqJjGWJppLHETG!zNaPXJjNRE:
G! htCOTO$H&w$3's it, od h waAsTG!avrndy-ucchouglahVZY
